# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [141]:
# Load the libraries as required.

%load_ext dotenv
%dotenv 
import os
import sys
sys.path.append(os.getenv('SRC_DIR'))

# Standard libraries
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import make_scorer, mean_absolute_error
from sklearn.ensemble import BaggingRegressor
import pickle
import shap
import matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [2]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [9]:
df_features = fires_dt.drop(columns= ["area"]) 
df_features.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 12 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
dtypes: float64(7), int64(3), object(2)
memory usage: 48.6+ KB


In [106]:
df_target = fires_dt["area"]
df_target.info()

<class 'pandas.core.series.Series'>
RangeIndex: 517 entries, 0 to 516
Series name: area
Non-Null Count  Dtype  
--------------  -----  
517 non-null    float64
dtypes: float64(1)
memory usage: 4.2 KB


# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [107]:



Preproc1 = ColumnTransformer(
    transformers=[
        ('numeric_transfomer', StandardScaler(), ['coord_x', 'coord_y','ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain'] ),
        ('onehot', OneHotEncoder(handle_unknown='ignore'), ['month', 'day']), 
    ], remainder='passthrough'
)

Preproc1

ColumnTransformer(remainder='passthrough',
                  transformers=[('numeric_transfomer', StandardScaler(),
                                 ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc',
                                  'isi', 'temp', 'rh', 'wind', 'rain']),
                                ('onehot',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['month', 'day'])])

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [108]:
Preproc2 = ColumnTransformer(
    transformers=[
        ('numeric_transfomer', StandardScaler(), ['coord_x', 'coord_y','ffmc', 'dmc', 'dc', 'isi','rain'] ),
        ('onehot', OneHotEncoder(handle_unknown='ignore'), ['month', 'day']),
        ('nonlineartransform', PowerTransformer(method='yeo-johnson'),['temp', 'rh', 'wind']) 
    ], remainder='passthrough'
)
Preproc2

ColumnTransformer(remainder='passthrough',
                  transformers=[('numeric_transfomer', StandardScaler(),
                                 ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc',
                                  'isi', 'rain']),
                                ('onehot',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['month', 'day']),
                                ('nonlineartransform', PowerTransformer(),
                                 ['temp', 'rh', 'wind'])])

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [109]:
scoring=["r2", "neg_mean_absolute_error", "neg_root_mean_squared_error"]

X_train, X_test, Y_train, Y_test = train_test_split(df_features, df_target, test_size = 0.2, random_state = 42)

In [110]:
# Pipeline A = preproc1 + baseline

PipelineA = Pipeline(
    [
        ('preprocessing', Preproc1), 
        ('regressor',KNeighborsRegressor(n_neighbors=3))
    ]
)
PipelineA



Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', KNeighborsRegressor(n_neighbors=3))])

In [111]:
res_pipelineA_dict = cross_validate(PipelineA, X_train, Y_train, cv = 5,scoring= scoring, return_train_score=True)
#res_pipelineA_dict
res_pipelineA = pd.DataFrame(res_pipelineA_dict).assign(experiment = 1)
res_pipelineA
res_pipelineA.mean()

fit_time                              0.010461
score_time                            0.010897
test_r2                              -1.985059
train_r2                              0.233166
test_neg_mean_absolute_error        -18.657061
train_neg_mean_absolute_error       -13.251721
test_neg_root_mean_squared_error    -51.912349
train_neg_root_mean_squared_error   -39.355447
experiment                            1.000000
dtype: float64

In [112]:
# Pipeline B = preproc2 + baseline

PipelineB = Pipeline(
    [
        ('preprocessing', Preproc2), 
        ('regressor',KNeighborsRegressor(n_neighbors=3))
    ]
)
PipelineB


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi',
                                                   'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day']),
                                                 ('nonlineartransform',
                                                  PowerTransformer(),
                                                  ['temp', 'rh', 'wind'])])),
                ('regressor', KNeighborsRegressor(n_neighbors=3))])

In [113]:
res_pipelineB_dict = cross_validate(PipelineB, X_train, Y_train, cv = 5, scoring = scoring,return_train_score=True)

res_pipelineB = pd.DataFrame(res_pipelineB_dict).assign(experiment = 2)
res_pipelineB
res_pipelineB.mean()

fit_time                              0.020631
score_time                            0.011570
test_r2                              -1.990250
train_r2                              0.213226
test_neg_mean_absolute_error        -19.049642
train_neg_mean_absolute_error       -13.582376
test_neg_root_mean_squared_error    -52.017554
train_neg_root_mean_squared_error   -39.899517
experiment                            2.000000
dtype: float64

In [114]:
# Pipeline C = preproc1 + advanced model
PipelineC = Pipeline(
    [
        ('preprocessing', Preproc1), 
        ('regressor',BaggingRegressor(n_estimators=50, random_state=42))
    ]
)
PipelineC



Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor',
                 BaggingRegressor(n_estimators=50, random_state=42))])

In [115]:
res_pipelineC_dict = cross_validate(PipelineC, X_train, Y_train, cv = 5, scoring = scoring,return_train_score=True)

res_pipelineC = pd.DataFrame(res_pipelineC_dict).assign(experiment = 3)
res_pipelineC
res_pipelineC.mean()

fit_time                              0.188713
score_time                            0.010328
test_r2                              -2.166807
train_r2                              0.810415
test_neg_mean_absolute_error        -20.886561
train_neg_mean_absolute_error        -7.773931
test_neg_root_mean_squared_error    -49.875298
train_neg_root_mean_squared_error   -19.587631
experiment                            3.000000
dtype: float64

In [117]:
# Pipeline D = preproc2 + advanced model

PipelineD = Pipeline(
    [
        ('preprocessing', Preproc2), 
        ('regressor',BaggingRegressor(n_estimators=50, random_state=42))
    ]
)
PipelineD

    

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi',
                                                   'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day']),
                                                 ('nonlineartransform',
                                                  PowerTransformer(),
                                                  ['temp', 'rh', 'wind'])])),
                ('regressor',
                 BaggingRegressor(n_estimators=50, random_state=42))])

In [118]:
res_pipelineD_dict = cross_validate(PipelineD, X_train, Y_train, cv = 5, scoring = scoring,return_train_score=True)

res_pipelineD = pd.DataFrame(res_pipelineD_dict).assign(experiment = 4)
res_pipelineD
res_pipelineD.mean()

fit_time                              0.221837
score_time                            0.013216
test_r2                              -1.764361
train_r2                              0.814202
test_neg_mean_absolute_error        -20.535114
train_neg_mean_absolute_error        -7.702402
test_neg_root_mean_squared_error    -48.702477
train_neg_root_mean_squared_error   -19.422138
experiment                            4.000000
dtype: float64

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [119]:
param_grid = {
    "regressor__n_neighbors": range(1,100,3)
    }

In [128]:
grid_cv_pipelineA = GridSearchCV(
    estimator=PipelineA, 
    param_grid=param_grid, 
    scoring = scoring, 
    cv = 5,
    refit="neg_root_mean_squared_error"
    )
grid_cv_pipelineA.fit(X_train, Y_train)

print("Best Parameters:", grid_cv_pipelineA.best_params_)
print("Best NRMSE Score:", grid_cv_pipelineA.best_score_)
print("Best estimator :", grid_cv_pipelineA.best_estimator_)



Best Parameters: {'regressor__n_neighbors': 40}
Best NRMSE Score: -38.29523211958792
Best estimator : Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', KNeighborsRegressor(n_neighbors=40))])


In [129]:
grid_cv_pipelineB = GridSearchCV(
    estimator=PipelineB, 
    param_grid=param_grid, 
    scoring = scoring, 
    cv = 5,
    refit="neg_root_mean_squared_error")
grid_cv_pipelineB.fit(X_train, Y_train)

print("Best Parameters:", grid_cv_pipelineB.best_params_)
print("Best NRMSE Score:", grid_cv_pipelineB.best_score_)
print("Best estimator :", grid_cv_pipelineB.best_estimator_)

Best Parameters: {'regressor__n_neighbors': 37}
Best NRMSE Score: -38.255962200789824
Best estimator : Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi',
                                                   'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day']),
                                                 ('nonlineartransform',
                                                  PowerTransformer(),
                                                  ['temp', 'rh', 'wind'])])),
           

In [122]:
param_grid_advanced = {
    "regressor__n_estimators": range(1,50,5)
    }

In [130]:
grid_cv_pipelineC = GridSearchCV(
    estimator=PipelineC, 
    param_grid=param_grid_advanced, 
    scoring = scoring, 
    cv = 5,
    refit="neg_root_mean_squared_error")

grid_cv_pipelineC.fit(X_train, Y_train)

print("Best Parameters:", grid_cv_pipelineC.best_params_)
print("Best NRMSE Score:", grid_cv_pipelineC.best_score_)
print("Best estimator :", grid_cv_pipelineC.best_estimator_)

Best Parameters: {'regressor__n_estimators': 11}
Best NRMSE Score: -48.683282625797844
Best estimator : Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor',
                 BaggingRegressor(n_estimators=11, random_state=42))])


In [131]:
grid_cv_pipelineD = GridSearchCV(
    estimator=PipelineD, 
    param_grid=param_grid_advanced, 
    scoring = scoring, 
    cv = 5,
    refit = "neg_root_mean_squared_error")

grid_cv_pipelineD.fit(X_train, Y_train)


print("Best Parameters:", grid_cv_pipelineD.best_params_)
print("Best NRMSE Score:", grid_cv_pipelineD.best_score_)
print("Best estimator :", grid_cv_pipelineD.best_estimator_)

Best Parameters: {'regressor__n_estimators': 36}
Best NRMSE Score: -47.8598190893779
Best estimator : Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi',
                                                   'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day']),
                                                 ('nonlineartransform',
                                                  PowerTransformer(),
                                                  ['temp', 'rh', 'wind'])])),
            

# Evaluate

+ Which model has the best performance?

In [134]:
print("The model with Preproc2 along with knn regression performed better becaue it has the smallest neg_root_mean_squared_error  ")

The model with Preproc2 along with knn regression performed better becaue it has the smallest neg_root_mean_squared_error  


# Export

+ Save the best performing model to a pickle file.

In [136]:
best_model = grid_cv_pipelineB.best_estimator_

with open("best_model.pkl", "wb") as file:
    pickle.dump(best_model, file)

In [181]:
best_model

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transfomer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi',
                                                   'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day']),
                                                 ('nonlineartransform',
                                                  PowerTransformer(),
                                                  ['temp', 'rh', 'wind'])])),
                ('regressor', KNeighborsRegressor(n_neighbors=37))])

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [ ]:
#feature_names = best_model.named_steps['preprocessing'].get_feature_names_out()

#X_test_transform = pd.DataFrame(best_model.named_steps['preprocessing'].transform(X_test),columns=feature_names)
#feature_names = best_model.feature_names_in_
X_train_transform = pd.DataFrame(best_model.named_steps['preprocessing'].transform(X_train),columns=feature_names)

X_train_transform.info()

#missing_cols = set(best_model.feature_names_in_) - set(X_train_transform.columns)
#print("Missing Columns:", missing_cols)
#X_train_sample = shap.sample(X_train_transform, 100)

#best_model.fit(X_train, Y_train)
#print
#explainer = shap.KernelExplainer(
    #best_model.predict, 
    #X_train_transform)
   

#shap_values = explainer(data_transform)

#explainer = shap.KernelExplainer(best_model.predict,X_train_transform)

#shap_values = explainer.shap_values(X_test_transform,100)


ValueError: input_features is not equal to feature_names_in_

In [183]:
print("Expected Features:", best_model.feature_names_in_)

Expected Features: ['numeric_transfomer__coord_x' 'numeric_transfomer__coord_y'
 'numeric_transfomer__ffmc' 'numeric_transfomer__dmc'
 'numeric_transfomer__dc' 'numeric_transfomer__isi'
 'numeric_transfomer__rain' 'onehot__month_apr' 'onehot__month_aug'
 'onehot__month_dec' 'onehot__month_feb' 'onehot__month_jan'
 'onehot__month_jul' 'onehot__month_jun' 'onehot__month_mar'
 'onehot__month_may' 'onehot__month_nov' 'onehot__month_oct'
 'onehot__month_sep' 'onehot__day_fri' 'onehot__day_mon' 'onehot__day_sat'
 'onehot__day_sun' 'onehot__day_thu' 'onehot__day_tue' 'onehot__day_wed'
 'nonlineartransform__temp' 'nonlineartransform__rh'
 'nonlineartransform__wind']


In [184]:
print("Actual Features:", X_train_transform.columns.tolist())

Actual Features: ['numeric_transfomer__coord_x', 'numeric_transfomer__coord_y', 'numeric_transfomer__ffmc', 'numeric_transfomer__dmc', 'numeric_transfomer__dc', 'numeric_transfomer__isi', 'numeric_transfomer__rain', 'onehot__month_apr', 'onehot__month_aug', 'onehot__month_dec', 'onehot__month_feb', 'onehot__month_jan', 'onehot__month_jul', 'onehot__month_jun', 'onehot__month_mar', 'onehot__month_may', 'onehot__month_nov', 'onehot__month_oct', 'onehot__month_sep', 'onehot__day_fri', 'onehot__day_mon', 'onehot__day_sat', 'onehot__day_sun', 'onehot__day_thu', 'onehot__day_tue', 'onehot__day_wed', 'nonlineartransform__temp', 'nonlineartransform__rh', 'nonlineartransform__wind']


In [185]:
best_model.fit(X_train_transform, Y_train)

ValueError: A given column is not a column of the dataframe

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.